# IT594 Project: Eval, relevance and constraint violation on WANDS

Scores every system in `runs` on **relevance (NDCG@10)** and **color-constraint violation**, reported in **separate tables**.
Metrics come from `src/metrics.py`, the color checker from `src/color_checker.py`.

## 1. Setup and load data

In [1]:
import re, sys
import numpy as np, pandas as pd

sys.path.insert(0, "../src")
from metrics import ndcg_at_k, constraint_violation_rate
from color_checker import query_colors, color_check

PATH = "../data/WANDS/dataset/"   # see data/README.md
products = pd.read_csv(PATH + "product.csv", sep="\t")
queries  = pd.read_csv(PATH + "query.csv",   sep="\t")
labels   = pd.read_csv(PATH + "label.csv",   sep="\t")

GAIN = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
labels["gain"] = labels["label"].map(GAIN)
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in labels.groupby("query_id")}
qtext = dict(zip(queries["query_id"], queries["query"]))
all_q = list(queries["query_id"])
print(products.shape, queries.shape, labels.shape)

(42994, 9) (480, 3) (233448, 5)


## 2. Runs
BM25 over name + class + first 500 characters of description (same text as notebook 03).
Add other systems as `runs[name] = {query_id: [product_id, ...]}`.

In [2]:
from rank_bm25 import BM25Okapi

for c in ["product_name", "product_class", "product_description"]:
    products[c] = products[c].fillna("")
products["text"] = products["product_name"] + ". " + products["product_class"] + ". " + products["product_description"].str.slice(0, 500)
pids = products["product_id"].values

tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in products["text"]])

runs = {"bm25": {q: list(pids[np.argsort(-bm25.get_scores(tok(qtext[q])))[:100]]) for q in all_q}}
print({name: len(r) for name, r in runs.items()})

{'bm25': 480}


In [3]:
import os
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
EMB = "wands_emb.npy"     # same cache file as notebook 03
if os.path.exists(EMB):
    E = np.load(EMB)
else:
    E = model.encode(products["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMB, E)

Qe = model.encode([qtext[q] for q in all_q], normalize_embeddings=True)
S = Qe @ E.T
runs["dense"] = {q: list(pids[np.argsort(-S[i])[:100]]) for i, q in enumerate(all_q)}
print("dense done")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense done


In [4]:
from retrieval import rrf

runs["hybrid_rrf"] = {q: rrf([runs["bm25"][q], runs["dense"][q]]) for q in all_q}
print({name: len(r) for name, r in runs.items()})

{'bm25': 480, 'dense': 480, 'hybrid_rrf': 480}


## 3. Relevance: NDCG@10 (all queries)

In [5]:
K = 10

rel = pd.DataFrame({
    "NDCG@10": {n: np.mean([ndcg_at_k(r[q], qrels.get(q, {}), K) for q in all_q]) for n, r in runs.items()},
}).round(3)
rel

,NDCG@10
bm25,0.687
dense,0.697
hybrid_rrf,0.748


## 4. Constraint violation: color (color queries only)
Product color comes from `color` or `primarycolor` in `product_features`, with the product name as fallback.

In [6]:
def parse_features(s):
    d = {}
    if pd.isna(s):
        return d
    for pair in s.split("|"):
        if ":" in pair:
            k, v = pair.split(":", 1)
            d[k.strip().lower()] = v.strip()
    return d

feat = products["product_features"].apply(parse_features)
pcolor = dict(zip(products["product_id"], feat.apply(lambda d: d.get("color") or d.get("primarycolor") or "")))
pname  = dict(zip(products["product_id"], products["product_name"]))

LENIENT = True    # main mode; set False for the strict ablation
color_qids = [q for q in all_q if query_colors(qtext[q], LENIENT)]
print("queries naming a color:", len(color_qids))

def checker(qid, lenient=LENIENT):
    qcols = query_colors(qtext[qid], lenient)
    return lambda p: color_check(qcols, pcolor.get(p, ""), pname.get(p, ""), lenient)

constraint = pd.DataFrame({
    n: {"violation_rate (checkable)": np.nanmean([constraint_violation_rate(r[q], checker(q), K) for q in color_qids]),
        "violating share of top-10": np.mean([constraint_violation_rate(r[q], checker(q), K, among="k") for q in color_qids]),
        "n_color_queries": len(color_qids)}
    for n, r in runs.items()}).T.astype({"n_color_queries": int})
constraint.round(3)

queries naming a color: 44


,violation_rate (checkable),violating share of top-10,n_color_queries
bm25,0.316,0.195,44
dense,0.592,0.323,44
hybrid_rrf,0.416,0.220,44


`violation_rate (checkable)` averages over color queries that have at least one checkable result in the top-10
(queries with none are skipped). `violating share of top-10` divides by 10, so ambiguous and unknown results count as not violating.

## 5. Is the difference real? Paired bootstrap over color queries

In [7]:
from metrics import paired_bootstrap

def per_query_viol(run):
    return [constraint_violation_rate(run[q], checker(q), K) for q in color_qids]

def per_query_ndcg(run):
    return [ndcg_at_k(run[q], qrels.get(q, {}), K) for q in color_qids]

PAIRS = [("dense", "bm25"), ("hybrid_rrf", "bm25"), ("hybrid_rrf", "dense")]

print("Relevance: NDCG@10 difference on color queries")
for a, b in PAIRS:
    m, (lo, hi), n = paired_bootstrap(per_query_ndcg(runs[a]), per_query_ndcg(runs[b]))
    print(f"  {a:10s} - {b:10s}: {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={n})")

print("\nConstraint: violation-rate difference on color queries")
for a, b in PAIRS:
    m, (lo, hi), n = paired_bootstrap(per_query_viol(runs[a]), per_query_viol(runs[b]))
    print(f"  {a:10s} - {b:10s}: {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={n})")

Relevance: NDCG@10 difference on color queries
  dense      - bm25      : +0.027  95% CI [-0.046, +0.105]  (n=44)
  hybrid_rrf - bm25      : +0.111  95% CI [+0.069, +0.158]  (n=44)
  hybrid_rrf - dense     : +0.085  95% CI [+0.029, +0.139]  (n=44)

Constraint: violation-rate difference on color queries
  dense      - bm25      : +0.259  95% CI [+0.180, +0.342]  (n=36)
  hybrid_rrf - bm25      : +0.077  95% CI [+0.010, +0.141]  (n=37)
  hybrid_rrf - dense     : -0.180  95% CI [-0.242, -0.121]  (n=36)


In [8]:
for q in color_qids:
    print(q, qtext[q], "->", sorted(query_colors(qtext[q], LENIENT)))

3 turquoise pillows -> ['teal']
15 black 5 drawer dresser by guilford -> ['black']
22 light and navy blue decorative pillow -> ['blue']
23 stoneford end tables white and wood -> ['white']
31 burnt orange curtains -> ['orange']
32 dark gray dresser -> ['gray']
46 beach blue headboard -> ['blue']
49 moen matte black hooks -> ['black']
50 tufted chair with gold legs -> ['gold']
55 tollette teal outdoor rug -> ['teal']
56 royal blue counter height chairs -> ['blue']
80 7 draw white dresser -> ['white']
92 antique silver entry table -> ['silver']
97 regner power loom red -> ['red']
134 black hanging bookshelf -> ['black']
136 black fluffy stool -> ['black']
162 turquoise chair -> ['teal']
188 bedroom wall decor floral, multicolored with some teal (prints) -> ['teal']
194 black frame 22 x 36 -> ['black']
195 black freestanding tub  with faucet -> ['black']
199 blue faux throw with pillows -> ['blue']
201 delta trinsic double towel hook in champagne bronze -> ['bronze']
204 fleur de lis livin

In [9]:
print(len(color_qids))

44


In [10]:
with open("color_queries.txt", "w") as f:
    for q in color_qids:
        f.write(f"{q}\t{qtext[q]}\t{sorted(query_colors(qtext[q], LENIENT))}\n")

In [11]:
import color_checker
print(color_checker.__file__)
print(open(color_checker.__file__).readline())

d:\constraint-search\notebooks\../src\color_checker.py
"""color_checker.py  (v2.1)  Color-constraint checker shared by all notebooks.



In [12]:
print(color_checker.__doc__.splitlines()[0])

color_checker.py  (v2.1)  Color-constraint checker shared by all notebooks.


## 6. Error analysis: are the violations real?
Every top-10 mismatch for each system, how the relevance labels judged it, and a hand check of 20 dense violations.

In [13]:
import os

LABEL = {2: "Exact", 1: "Partial", 0: "Irrelevant"}

def violations(system):
    rows = []
    for q in color_qids:
        chk = checker(q)
        for rank, p in enumerate(runs[system][q][:K], 1):
            if chk(p) == "mismatch":
                rows.append({"system": system, "qid": q, "query": qtext[q], "rank": rank,
                             "product": pname[p][:70], "color_field": pcolor[p] or "(name)",
                             "label": LABEL.get(qrels.get(q, {}).get(p), "unjudged")})
    return pd.DataFrame(rows)

viol_df = pd.concat([violations(s) for s in runs], ignore_index=True)
pd.crosstab(viol_df["system"], viol_df["label"], margins=True)

label,Exact,Irrelevant,Partial,unjudged,All
system,,,,,
bm25,3,8,41,34,86
dense,8,1,72,61,142
hybrid_rrf,8,5,65,19,97
All,19,14,178,114,325


In [14]:
CHECK = "../results/dense_violation_check.csv"
if os.path.exists(CHECK):
    print("Already exported; not overwriting.")
else:
    d = viol_df[viol_df["system"] == "dense"]
    sample = d.sample(min(20, len(d)), random_state=0)[["query", "product", "color_field", "label", "rank"]]
    sample["my_verdict"] = ""
    sample.to_csv(CHECK, index=False)
    print("Saved", len(sample), "rows to", CHECK)

Saved 20 rows to ../results/dense_violation_check.csv


In [15]:
chk = pd.read_csv(CHECK)
done = chk.dropna(subset=["my_verdict"])
print(f"Checked by hand: {len(done)} dense violations")
print(done["my_verdict"].value_counts())
print(f"\nReal violations: {(done['my_verdict'] == 'real').mean():.0%}")

Checked by hand: 0 dense violations
Series([], Name: count, dtype: int64)

Real violations: nan%
